# Check 03 — Runtime Adapter

**Category:** Module smoke check (fast regression; companion to pytest, not a full tutorial).

**Purpose:** Prove PyPI adapter wheels load (`exo-adapter-openai`, `exo-adapter-echo`), adapter healthchecks succeed, session start and capabilities metadata work, and `run_turn` with `planned_tool_call` emits an exact `TOOL_INTENT`. This validates the **planned-tool injection path** — not live model-driven tool choice (no API key).

**Prerequisites:**
- Python **3.12+** with project deps installed (`pip install -r requirements.txt` from repo root)
- Kernel: project **`.venv`** (see `notebooks/README.md`)
- Run cells **top to bottom** (bootstrap cell sets `sys.path` automatically)
- **No API key** required — deterministic, in-process only

**Related tutorial:** `tutorial_02_openai_adapter.ipynb`

**Modules exercised:** `exo-adapter-openai`, `exo-adapter-echo` (PyPI wheels), `src/runtime/openai_agents_runtime` (shim re-export), `exo-brain-core-contracts` (via `src/schemas/events`)

**PASS means:** Echo and OpenAI healthchecks are `HEALTHY`; capabilities `provider_id` is `openai`; exactly one `TOOL_INTENT` matches the planned call fields (`call_id`, `tool_name`, arguments, risk tier, state-changing flag, run/session ids); final line `PASS: runtime adapter planned_tool_call emitted exact TOOL_INTENT`.

**Troubleshooting:** If `nest_asyncio` is missing in Jupyter, `pip install nest-asyncio` (listed in `requirements.txt`). Live OpenAI calls are **not** required here.

In [1]:
import pathlib
import sys

_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(_root))

import importlib
import importlib.util

_ADAPTER_WHEELS = (
    ("exo-brain-core-contracts", "exo_brain_core_contracts"),
    ("exo-brain-adapter-sdk", "exo_brain_adapter_sdk"),
    ("exo-adapter-echo", "exo_adapter_echo"),
    ("exo-adapter-openai", "exo_adapter_openai"),
)


def _print_adapter_wheels() -> None:
    for dist, module_name in _ADAPTER_WHEELS:
        if importlib.util.find_spec(module_name) is None:
            print(f"warn: {dist} not installed — pip install -r requirements.txt")
            continue
        mod = importlib.import_module(module_name)
        mod_file = (mod.__file__ or "").replace("\\", "/")
        if "site-packages" not in mod_file and "dist-packages" not in mod_file:
            raise RuntimeError(f"{dist} must be a PyPI wheel in site-packages, got {mod.__file__}")
        if "/eXo_adapters/" in mod_file:
            raise RuntimeError(
                f"{dist} must not load from eXo_adapters checkout — "
                f"pip install -r requirements.txt: {mod.__file__}"
            )
        print(f"{dist}:", mod.__file__)


_print_adapter_wheels()

from src.runtime.openai_agents_runtime import OpenAIAgentsRuntimeAdapter

assert OpenAIAgentsRuntimeAdapter.__module__.startswith("exo_adapter_openai."), (
    "OpenAIAgentsRuntimeAdapter must come from exo-adapter-openai (PyPI); "
    "reinstall: pip install -r requirements.txt"
)

from exo_adapter_echo.runtime import EchoRuntimeAdapter

assert EchoRuntimeAdapter.__module__.startswith("exo_adapter_echo."), (
    "EchoRuntimeAdapter must come from exo-adapter-echo (PyPI); "
    "reinstall: pip install -r requirements.txt"
)

print("OpenAIAgentsRuntimeAdapter module:", OpenAIAgentsRuntimeAdapter.__module__)
print("EchoRuntimeAdapter module:", EchoRuntimeAdapter.__module__)

from src.runtime.capability_map import HealthState
from src.schemas.events import RuntimeEventType
from src.runtime.openai_agents_runtime import OpenAIAgentsRuntimeAdapter

exo-brain-core-contracts: /home/razvansavin/Projects/eXo-brain/.venv/lib/python3.12/site-packages/exo_brain_core_contracts/__init__.py
exo-brain-adapter-sdk: /home/razvansavin/Projects/eXo-brain/.venv/lib/python3.12/site-packages/exo_brain_adapter_sdk/__init__.py
exo-adapter-echo: /home/razvansavin/Projects/eXo-brain/.venv/lib/python3.12/site-packages/exo_adapter_echo/__init__.py
exo-adapter-openai: /home/razvansavin/Projects/eXo-brain/.venv/lib/python3.12/site-packages/exo_adapter_openai/__init__.py
OpenAIAgentsRuntimeAdapter module: exo_adapter_openai.runtime
EchoRuntimeAdapter module: exo_adapter_echo.runtime


In [2]:
import asyncio

async def _run_check():
    echo = EchoRuntimeAdapter()
    echo_health = await echo.healthcheck()
    assert echo_health.state == HealthState.HEALTHY, echo_health.state
    print("echo health:", echo_health.state.value, echo_health.reason)

    adapter = OpenAIAgentsRuntimeAdapter()
    handle = await adapter.start_session("sess_runtime_nb", {"agent_id": "runtime-nb"})
    assert handle.session_id == "sess_runtime_nb"

    health = await adapter.healthcheck()
    assert health.state == HealthState.HEALTHY, health.state
    caps = adapter.get_capabilities()
    assert caps.provider_id == "openai", caps.provider_id
    print("health:", health.state.value, health.reason)
    print("capabilities provider:", caps.provider_id)

    context = {
        "run_id": "run_runtime_nb",
        "job_id": "job_runtime_nb",
        "task_id": "task_runtime_nb",
        "agent_id": "agent_runtime_nb",
        "planned_tool_call": {
            "call_id": "tc_runtime_nb",
            "tool_name": "fake_tool",
            "arguments": {"x": 1},
            "risk_tier": "low",
            "is_state_changing": False,
        },
    }
    events = []
    async for event in adapter.run_turn("sess_runtime_nb", "hello", context):
        events.append(event)
        print(event.event_type.value, event.payload)

    event_types = [e.event_type for e in events]
    assert event_types == [RuntimeEventType.TOOL_INTENT], event_types

    intent = events[0]
    assert intent.tool_call is not None
    tc = intent.tool_call
    assert tc.call_id == "tc_runtime_nb"
    assert tc.tool_name == "fake_tool"
    assert tc.arguments == {"x": 1}
    assert tc.risk_tier.value == "low"
    assert tc.is_state_changing is False
    assert intent.run_id == "run_runtime_nb"
    assert intent.session_id == "sess_runtime_nb"
    print("tool_intent call_id:", tc.call_id)
    print("tool_intent tool_name:", tc.tool_name)
    print("tool_intent arguments:", tc.arguments)
    print("tool_intent risk_tier:", tc.risk_tier.value)
    print("tool_intent is_state_changing:", tc.is_state_changing)
    print("PASS: runtime adapter planned_tool_call emitted exact TOOL_INTENT")

# Works in both async-native kernels and standard synchronous kernels
try:
    loop = asyncio.get_running_loop()
    import nest_asyncio
    nest_asyncio.apply()
    loop.run_until_complete(_run_check())
except RuntimeError:
    asyncio.run(_run_check())

echo health: healthy echo-adapter-initialized
health: healthy adapter-initialized
capabilities provider: openai
tool_intent {}
tool_intent call_id: tc_runtime_nb
tool_intent tool_name: fake_tool
tool_intent arguments: {'x': 1}
tool_intent risk_tier: low
tool_intent is_state_changing: False
PASS: runtime adapter planned_tool_call emitted exact TOOL_INTENT
